# Scrapping paso a paso de concellos


## Paso 1 . Comprobar descarga

### Pasos para comprobar con Robotfileparser 
1. Realizar una consulta get para recoger el contenido del archivo txt a tratar con las cabeceras que hemos definido previamente 
    * Hacemos una comprobación previa para comprobar que la url a tratar existe , esto dentro de un try catch 
2. Utilizamos RobotFileParser para interpretar las reglas que contiene el archivo txt
3. Debido a la existencia de líneas BOM(indicando codificación) en el txt , procedemos a su eliminación para una lectura más limpia y que no interfiera en la interpretación de las líneas del archivo.
4. Buscamos meidante un bucle for si la url a analizar de concellos permite a nuetsro User-Agent realizar un scrapping, esto se realiza por un can_fetch 
    *  Comprueba si la URL coincide con alguna ruta prohibida robots

In [ ]:
import time
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://en.wikipedia.org/"
CABECEIRAS = {"User-Agent": "E4_Concellos (carolinamesacurso@gmail.com)"}
PAUSA = 1.0                 

RAW = Path("../datos/raw")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

In [2]:
# Consultamos el contenido de robots.txt
respuesta_robots = requests.get(urljoin(BASE, "robots.txt"),headers=CABECEIRAS,timeout=10)

print("Código:", respuesta_robots.status_code)
try:
    respuesta_robots.raise_for_status()
except requests.HTTPError as e:
    print("HTTPError:", e)

print(respuesta_robots.text)

# Creamos el analizador de robots.txt
robots = RobotFileParser(urljoin(BASE, "robots.txt"))

# Eliminamos el BOM y cargamos las reglas
robots.parse(respuesta_robots.text.lstrip("\ufeff").splitlines())

# Comprobamos si podemos acceder a la página
for ruta in ["wiki/List_of_municipalities_in_A_Coru%C3%B1a"]:
    print(f"{ruta:55} permitido? "f"{robots.can_fetch(CABECEIRAS['User-Agent'], urljoin(BASE, ruta))}")

# Consultamos el tiempo de espera
print("Crawl-delay:", robots.crawl_delay(CABECEIRAS["User-Agent"]))

Código: 200
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /

# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
Disallow:

# Crawlers that are kind enough to obey, but which we'd rather not have
# unless they're feeding search engines.
User-agent: UbiCrawler
Disallow: /

User-agent: DOC
Disallow: /

User-agent: Zao
Disallow: /

# Some bots are known to be trouble, particularly those designed to copy
# entire sites. Please obey robots.txt.
User-agent: sitecheck.internetseer.com
Disallow: /



    
### Que permite la licencia y datos personales que existen en ella
* La licencia que nos permite leer las páginas reglamentarias de wikipedia se llama  'CC BY-SA 4.0.', la cual nos permite modificar , leer , copiar y reutilizar el contenido .
* Podemos observar por los datos mostrados anteriormente que tenemos ciertos agentes que no están permitidos y ciertas páginas las cuales no podemos scrapear.
* En el caso de la página de concellos vemos que no existen datos personales ya que solo observamso datos de los concellos mostrados públicamente.

Disallow: /w/
Disallow: /api/
Disallow: /trap/
Disallow: /wiki/Special:

# Paso 2 . Descarga

### Realizar comprobaciones adicionales
* Comprobamos que la codificación de la url a tratar es válida o la indicada para trabajar . En este caso comprobamos que es utf-8 asi que estaría correcto.

In [3]:
URL_municipios = urljoin(BASE,"wiki/List_of_municipalities_in_A_Coru%C3%B1a")
resposta = requests.get(URL_municipios, headers=CABECEIRAS, timeout=10)
print("Código de estado:", resposta.status_code)
print("Content-Type    :", resposta.headers.get("Content-Type"))
print("Codificación    :", resposta.encoding)
print()
print(resposta.text[:100])

Código de estado: 200
Content-Type    : text/html; charset=UTF-8
Codificación    : UTF-8

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-la


In [4]:
import time

# Tiempo de espera entre peticiones (segundos)
PAUSA = 2

# Función para descargar y guardar una página
def descargar(url, destino):
    if destino.exists():
        return "ya existía, se cancela la descarga"

    r = requests.get(url, headers=CABECEIRAS, timeout=10)
    r.raise_for_status()
    destino.write_bytes(r.content)
    time.sleep(PAUSA)

    return "descargado"

# Definimos el nombre del archivo
fichero = RAW / "municipios_coruna.html"

# Descargamos la página
print(fichero.name, descargar(URL_municipios, fichero))

municipios_coruna.html ya existía, se cancela la descarga


## Paso 3 . Extracción

In [5]:
sopa = BeautifulSoup(fichero.read_bytes(), "html.parser") 
# Buscamos todas las tablas
tablas = sopa.find_all("table")

tabla_municipios = None

for tabla in tablas:
 
    cabeceiras = [
        th.get_text(" ", strip=True)
        for th in tabla.find_all("th")
    ]
    for cabeceira in cabeceiras:
        if "Population (2024)" in cabeceira:
            tabla_municipios = tabla
            break
    if tabla_municipios is not None:
        break

print("Tabla encontrada:", tabla_municipios is not None)
print(tabla_municipios)


Tabla encontrada: True
<table class="wikitable sortable" id="mwvQ">
<tbody id="mwvg"><tr id="mwvw"><th id="mwwA"><b id="mwwQ">Name</b></th>
<th id="mwwg"><b id="mwww">Population (2024)<sup about="#mwt79" class="mw-ref reference" data-mw='{"name":"ref","attrs":{},"body":{"id":"mw-reference-text-cite_note-20"}}' id="cite_ref-20" rel="dc:references" typeof="mw:Extension/ref"><a href="#cite_note-20" id="mwxA"><span class="mw-reflink-text" id="mwxQ"><span class="cite-bracket" id="mwxg">[</span>20<span class="cite-bracket" id="mwxw">]</span></span></a></sup></b></th></tr>
<tr id="mwyA">
<td id="mwyQ"><a href="https://en.wikipedia.org/wiki/Abegondo" id="mwyg" rel="mw:WikiLink" title="Abegondo">Abegondo</a></td>
<td id="mwyw">5,578</td></tr>
<tr id="mwzA">
<td id="mwzQ"><a class="mw-redirect" href="https://en.wikipedia.org/wiki/Ames,_A_Coruña" id="mwzg" rel="mw:WikiLink" title="Ames, A Coruña">Ames</a></td>
<td id="mwzw">32,812</td></tr>
<tr id="mw0A">
<td id="mw0Q"><a href="https://en.wikiped

In [28]:
def extraer_municipios(fichero):

    sopa = BeautifulSoup(fichero.read_bytes(), "html.parser")

    tablas = sopa.find_all("table")
    taboa = None

    for tabla in tablas:
        cabeceiras = [
            th.get_text(" ", strip=True)
            for th in tabla.find_all("th")
        ]

        for cabeceira in cabeceiras:
            if "Population (2024)" in cabeceira:
                taboa = tabla
                break

        if taboa is not None:
            break

    filas = []

    for tr in taboa.find_all("tr"):
        celdas = tr.find_all("td")

        if len(celdas) == 2:
            filas.append({
                "concello": celdas[0].get_text(strip=True),
                "poblacion": celdas[1].get_text(strip=True)
            })

    return filas

filas = extraer_municipios(fichero)

municipios = pd.DataFrame(filas)

print(municipios)


      concello poblacion
0     Abegondo     5,578
1         Ames    32,812
2       Aranga     1,799
3         Ares     6,156
4      Arteixo    34,038
..         ...       ...
88       Vedra     4,942
89  Vilarmaior     1,252
90  Vilasantar     1,229
91    Vimianzo     6,808
92         Zas     4,261

[93 rows x 2 columns]


## Paso 4. Limpieza

In [29]:
print("Valores con coma:", municipios["poblacion"].str.contains(",").sum())
print("Valores con punto:", municipios["poblacion"].str.contains(r"\.").sum())


Valores con coma: 92
Valores con punto: 1


In [30]:
print(municipios["poblacion"].value_counts().head(10))

poblacion
5,578     1
32,812    1
1,799     1
6,156     1
34,038    1
5,887     1
3,291     1
6,986     1
13,261    1
1.801     1
Name: count, dtype: int64


In [31]:
def numero_es(serie):
    return pd.to_numeric(
        serie.str.replace(",", "", regex=False).str.replace(".", "", regex=False).astype(int),
        errors="coerce",
    )

for col in ["poblacion"]:
    municipios[col] = numero_es(municipios[col])
print(municipios.dtypes)
print("Municipios sen población:", municipios["poblacion"].isna().sum())


concello       str
poblacion    int64
dtype: object
Municipios sen población: 0


In [32]:
print(municipios)


      concello  poblacion
0     Abegondo       5578
1         Ames      32812
2       Aranga       1799
3         Ares       6156
4      Arteixo      34038
..         ...        ...
88       Vedra       4942
89  Vilarmaior       1252
90  Vilasantar       1229
91    Vimianzo       6808
92         Zas       4261

[93 rows x 2 columns]


## Paso 5. Validación

In [33]:
municipios.describe(include="all")

,concello,poblacion
count,93,93.000000
unique,93,NaN
top,Abegondo,NaN
freq,1,NaN
mean,NaN,12132.473118
std,NaN,28528.028535
min,NaN,1004.000000
25%,NaN,2979.000000
50%,NaN,5195.000000
75%,NaN,9064.000000


In [34]:
print("Número de concellos:", len(municipios))
print("Concellos duplicados:", municipios["concello"].duplicated().sum())

Número de concellos: 93
Concellos duplicados: 0


In [35]:
municipios["poblacion"].sum()
#A partir de 2024 Según el censo español, la provincia es la duodécima más grande por población , con 1.128.320 habitantes

np.int64(1128320)

In [45]:
texto_habitantes = ""

parrafos = sopa.find_all("p")

for parrafo in parrafos:
    texto = parrafo.get_text(" ", strip=True)

    if "inhabitants" in texto.lower():
        texto_habitantes = texto
        break

print(texto_habitantes)

A Coruña is a province in the autonomous community of Galicia , Spain, that is divided into 93 municipalities . As of the 2024 [ update ] Spanish census, the province is the 12th largest by population , with 1,128,320 inhabitants, [ 1 ] and the 32nd largest by land area, spanning 7,950 square kilometres (3,070 sq mi) . [ 2 ] Municipalities are the basic local political division in Spain [ 3 ] and can only belong to one province. [ 4 ] They enjoy a large degree of autonomy in their local administration, being in charge of tasks such as urban planning, water supply, lighting, roads, local police, and firefighting. [ 5 ]


In [46]:
poblacion_provincia = texto_habitantes.split("inhabitants")[0].split()[-1]

print(poblacion_provincia)

1,128,320


In [17]:
tabla_readhtml = pd.read_html(StringIO(fichero.read_text(encoding="utf-8")),decimal=",",thousands=".", na_values=["-"]) #decimal=".", thousands=",",
print("Táboas atopadas:", len(tabla_readhtml))
tabla_readhtml[0].head()

Táboas atopadas: 2


,Name,Population (2024)[20]
0,Abegondo,5.578
1,Ames,32.812
2,Aranga,1.799
3,Ares,6.156
4,Arteixo,34.038


In [18]:
tabla_readhtml = pd.read_html(
    StringIO(fichero.read_text(encoding="utf-8")),
    converters={1: str},
    na_values=["—"]
)

municipios_readhtml = tabla_readhtml[0]

In [19]:
municipios_readhtml["Population (2024)[20]"] = (
    municipios_readhtml["Population (2024)[20]"]
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
)

In [20]:
municipios_readhtml["Population (2024)[20]"] = (
    pd.to_numeric(
        municipios_readhtml["Population (2024)[20]"],
        errors="coerce"
    ).astype("Int64")
)

In [21]:
municipios_readhtml["Population (2024)[20]"].head(20).tolist()

[5578,
 32812,
 1799,
 6156,
 34038,
 5887,
 3291,
 6986,
 13261,
 1801,
 19018,
 4200,
 8197,
 4098,
 3296,
 5098,
 24781,
 1182,
 31595,
 3691]

In [22]:
municipios_readhtml.describe(include="all")

,Name,Population (2024)[20]
count,93,93.0
unique,93,<NA>
top,Abegondo,<NA>
freq,1,<NA>
mean,NaN,12132.473118
std,NaN,28528.028535
min,NaN,1004.0
25%,NaN,2979.0
50%,NaN,5195.0
75%,NaN,9064.0


In [23]:
municipios_readhtml["Population (2024)[20]"].sum()


np.int64(1128320)

## Paso 6 . Preguntas

In [ ]:

municipios.sort_values("poblacion", ascending=False)[0:5]




,concello,poblacion
29,A Coruña,249261
77,Santiago de Compostela,99536
35,Ferrol,64218
53,Narón,39285
57,Oleiros,38333


In [77]:
menos5000=municipios[municipios["poblacion"] < 5000]["poblacion"].count()
menos5000

np.int64(43)

In [78]:
porcentaje = (menos5000 / municipios.shape[0]) * 100

print(f"Porcentaje de concellos: {porcentaje:.2f}%")

Porcentaje de concellos: 46.24%


In [82]:
poblaciontotal3=municipios.sort_values("poblacion", ascending=False)["poblacion"][0:3].sum()
poblaciontotal3


np.int64(413015)

In [83]:
porcentaje3=(poblaciontotal3 / municipios["poblacion"].sum()) * 100
print(f"Porcentaje de la población en los 3 concellos con más habitantes: {porcentaje3:.2f}%")

Porcentaje de la población en los 3 concellos con más habitantes: 36.60%


In [ ]:
mitad_poblacion = municipios["poblacion"].sum() / 2
poblacionCalculada=mitad_poblacion -poblaciontotal3
poblacionCalculada #nos faltarían esta cantidad de habitantes para llegar a la mitad de la población de la provincia


np.float64(151145.0)

In [88]:
municipios_ordenados = municipios.sort_values(
    "poblacion", ascending=False
)

In [89]:
municipios_ordenados["poblacion_acumulada"] = municipios_ordenados["poblacion"].cumsum()

In [91]:
municipiosascumulados=municipios_ordenados[
    municipios_ordenados["poblacion_acumulada"] >= mitad_poblacion
].head(1)
municipiosascumulados

,concello,poblacion,poblacion_acumulada
18,Carballo,31595,589078


In [94]:
concellos_necesarios = (
    municipios_ordenados["poblacion_acumulada"] < mitad_poblacion
).sum() + 1

concellos_necesarios
municipios_ordenados.head(concellos_necesarios)

,concello,poblacion,poblacion_acumulada
29,A Coruña,249261,249261
77,Santiago de Compostela,99536,348797
35,Ferrol,64218,413015
53,Narón,39285,452300
57,Oleiros,38333,490633
4,Arteixo,34038,524671
1,Ames,32812,557483
18,Carballo,31595,589078


In [95]:
# Guardar datos limpios en csv
municipios.to_csv("../datos/clean/municipios_coruna_limpio.csv", index=False, sep=";", encoding="utf-8")

In [ ]:

# --------------------------------------------------
# 3. DESCARGAR LA PÁGINA SI ESTÁ PERMITIDA
# --------------------------------------------------

# if permitido:

#     time.sleep(PAUSA)

#     respuesta = requests.get(
#         URL,
#         headers=CABECEIRAS,
#         timeout=10
#     )

#     respuesta.raise_for_status()

#     print("Estado HTTP:", respuesta.status_code)

#     # Guardar HTML original
#     archivo_html = RAW / "municipios_coruna.html"

#     archivo_html.write_text(
#         respuesta.text,
#         encoding="utf-8"
#     )

#     print("HTML guardado en:", archivo_html)

# else:
#     print("El rastreo de esta página no está permitido.")